In [1]:
import os
import shutil
import kagglehub


os.makedirs("data", exist_ok=True)

# Download dataset
path = kagglehub.dataset_download(
    "nickfratto/pacs-dataset",
    output_dir="data"
)

print("Path to dataset files:", path)

Path to dataset files: data


In [2]:
import os
os.listdir("../workspace/data/pacs_data/pacs_data/art_painting")

['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']

In [3]:
import os
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split


# ============================================================
# Configuration
# ============================================================

DATA_ROOT = "../workspace/data/pacs_data/pacs_data"

SOURCE_DOMAINS = [
    "photo",
    "art_painting",
    "cartoon",
]

TARGET_DOMAIN = "sketch"

SEED = 6304
BATCH_SIZE = 32
NUM_WORKERS = 4


# ============================================================
# Reproducibility
# ============================================================

torch.manual_seed(SEED)
np.random.seed(SEED)

generator = torch.Generator()
generator.manual_seed(SEED)


# ============================================================
# Transforms
# ============================================================

# Training augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    # ImageNet normalization -- suitable for ImageNet
    # pretrained backbones such as ResNet.
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Validation / test / target transform
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# Create stratified split for one source domain
# ============================================================

def create_source_split(domain):
    domain_path = os.path.join(DATA_ROOT, domain)

    # Dataset used for training
    train_dataset = datasets.ImageFolder(
        domain_path,
        transform=train_transform
    )

    # Same images but evaluation transforms
    eval_dataset = datasets.ImageFolder(
        domain_path,
        transform=eval_transform
    )

    labels = np.array(train_dataset.targets)
    indices = np.arange(len(labels))

    # --------------------------------------------------------
    # Stratified 80/20 split
    # --------------------------------------------------------

    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED,
        stratify=labels
    )

    train_subset = Subset(
        train_dataset,
        train_idx
    )

    val_subset = Subset(
        eval_dataset,
        val_idx
    )

    return train_subset, val_subset, eval_dataset


# ============================================================
# Split each SOURCE DOMAIN independently
# ============================================================

source_train_sets = []
source_val_sets = []
source_test_sets = []

for domain in SOURCE_DOMAINS:

    train_set, val_set, test_set = create_source_split(domain)

    source_train_sets.append(train_set)
    source_val_sets.append(val_set)
    source_test_sets.append(test_set)

    print(
        f"{domain:15s} | "
        f"train = {len(train_set):5d} | "
        f"val = {len(val_set):5d} | "
        f"total = {len(test_set):5d}"
    )


# ============================================================
# Combine the three source domains
# ============================================================

source_train_dataset = ConcatDataset(source_train_sets)
source_val_dataset = ConcatDataset(source_val_sets)

# Evaluation over all source images
source_test_dataset = ConcatDataset(source_test_sets)


# ============================================================
# Target domain: SKETCH
# ============================================================

target_path = os.path.join(DATA_ROOT, TARGET_DOMAIN)

target_dataset = datasets.ImageFolder(
    target_path,
    transform=eval_transform
)


# ============================================================
# DataLoaders
# ============================================================

train_loader_source = DataLoader(
    source_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=generator
)


val_loader_source = DataLoader(
    source_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


test_loader_source = DataLoader(
    source_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


target_loader = DataLoader(
    target_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# ============================================================
# Sanity checks
# ============================================================

print("\nDataset sizes")
print("-" * 45)
print(f"Source train : {len(source_train_dataset)}")
print(f"Source val   : {len(source_val_dataset)}")
print(f"Source test  : {len(source_test_dataset)}")
print(f"Target       : {len(target_dataset)}")

print("\nTarget classes:")
print(target_dataset.classes)

print("\nClass -> index:")
print(target_dataset.class_to_idx)

photo           | train =  1336 | val =   334 | total =  1670
art_painting    | train =  1638 | val =   410 | total =  2048
cartoon         | train =  1875 | val =   469 | total =  2344

Dataset sizes
---------------------------------------------
Source train : 4849
Source val   : 1213
Source test  : 6062
Target       : 3929

Target classes:
['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']

Class -> index:
{'dog': 0, 'elephant': 1, 'giraffe': 2, 'guitar': 3, 'horse': 4, 'house': 5, 'person': 6}


In [4]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


class PACSResNet18(nn.Module):
    """
    ResNet-18 for PACS.

    Architecture:
        Image
          ↓
        ResNet-18 backbone
          ↓
        512-dimensional feature vector
          ↓
        Linear classifier
          ↓
        7-class logits

    The 512-D features can later be used for DAN/MMD alignment.
    """

    def __init__(self, num_classes=7):
        super().__init__()

        # ----------------------------------------------------
        # Pretrained ResNet-18
        # ----------------------------------------------------
        weights = ResNet18_Weights.IMAGENET1K_V1
        backbone = resnet18(weights=weights)

        # Everything except the original ImageNet classifier.
        #
        # Output after avgpool:
        #     [batch_size, 512, 1, 1]
        self.feature_extractor = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        # ----------------------------------------------------
        # PACS classifier
        # ----------------------------------------------------
        self.classifier = nn.Linear(
            in_features=512,
            out_features=num_classes
        )


    def forward(self, x, return_features=False):

        # [B, 3, 224, 224]
        features = self.feature_extractor(x)

        # [B, 512, 1, 1] -> [B, 512]
        features = torch.flatten(features, 1)

        # [B, 512] -> [B, 7]
        logits = self.classifier(features)

        if return_features:
            return logits, features

        return logits

In [5]:
SEED = 6304

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = PACSResNet18(num_classes=7).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:30<00:00, 1.52MB/s]


In [6]:
import copy
from sklearn.metrics import f1_score

# Domain-balanced loaders: 8 samples from each source domain
source_loaders = [
    DataLoader(
        ds, batch_size=8, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    for ds in source_train_sets
]

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

def evaluate(loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    acc = np.mean(np.array(preds) == np.array(labels))
    f1 = f1_score(labels, preds, average="macro")
    return acc, f1


best_f1 = -1
best_state = None
patience = 5
bad_epochs = 0

for epoch in range(30):
    model.train()

    iters = [iter(loader) for loader in source_loaders]
    steps = max(len(loader) for loader in source_loaders)

    for _ in range(steps):
        xs, ys = [], []

        for i, loader in enumerate(source_loaders):
            try:
                x, y = next(iters[i])
            except StopIteration:
                iters[i] = iter(loader)
                x, y = next(iters[i])

            xs.append(x)
            ys.append(y)

        x = torch.cat(xs).to(device)
        y = torch.cat(ys).to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    val_acc, val_f1 = evaluate(val_loader_source)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "source_only_erm.pt")
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping.")
        break


# Load best checkpoint
model.load_state_dict(best_state)

print("\nPer-source validation performance")
for domain, val_set in zip(SOURCE_DOMAINS, source_val_sets):
    loader = DataLoader(
        val_set, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS
    )
    acc, f1 = evaluate(loader)
    print(f"{domain:15s} | Acc: {acc:.4f} | Macro-F1: {f1:.4f}")

target_acc, target_f1 = evaluate(target_loader)

print("\nTarget (sketch)")
print(f"Accuracy: {target_acc:.4f}")
print(f"Macro-F1: {target_f1:.4f}")

Epoch 01 | Val Acc: 0.9340 | Val Macro-F1: 0.9319
Epoch 02 | Val Acc: 0.9340 | Val Macro-F1: 0.9322
Epoch 03 | Val Acc: 0.9390 | Val Macro-F1: 0.9395
Epoch 04 | Val Acc: 0.9340 | Val Macro-F1: 0.9345
Epoch 05 | Val Acc: 0.9283 | Val Macro-F1: 0.9282
Epoch 06 | Val Acc: 0.9398 | Val Macro-F1: 0.9414
Epoch 07 | Val Acc: 0.9398 | Val Macro-F1: 0.9405
Epoch 08 | Val Acc: 0.9406 | Val Macro-F1: 0.9416
Epoch 09 | Val Acc: 0.9299 | Val Macro-F1: 0.9318
Epoch 10 | Val Acc: 0.9340 | Val Macro-F1: 0.9355
Epoch 11 | Val Acc: 0.9324 | Val Macro-F1: 0.9305
Epoch 12 | Val Acc: 0.9406 | Val Macro-F1: 0.9409
Epoch 13 | Val Acc: 0.9406 | Val Macro-F1: 0.9397
Early stopping.

Per-source validation performance
photo           | Acc: 0.9671 | Macro-F1: 0.9599
art_painting    | Acc: 0.9146 | Macro-F1: 0.9168
cartoon         | Acc: 0.9446 | Macro-F1: 0.9455

Target (sketch)
Accuracy: 0.6103
Macro-F1: 0.5948


In [7]:
# Save ERM baseline checkpoint
torch.save(model.state_dict(), "erm-baseline.pt")
print("Saved checkpoint: erm-baseline.pt")

Saved checkpoint: erm-baseline.pt


# DAN -MMD Alignment

In [8]:
import copy
from itertools import cycle
from sklearn.metrics import f1_score

# ------------------------------------------------------------
# Fresh DAN model — DO NOT load the ERM baseline
# ------------------------------------------------------------
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dan_model = PACSResNet18(num_classes=7).to(device)

# 8 samples from each source domain = 24 source samples
source_loaders = [
    DataLoader(
        ds,
        batch_size=8,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    for ds in source_train_sets
]

# 24 unlabeled target samples
target_train_loader = DataLoader(
    target_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    dan_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# MMD with three RBF kernels
# ------------------------------------------------------------
def mmd_loss(source, target):
    z = torch.cat([source, target], dim=0)

    # Pairwise squared feature distances
    d2 = torch.cdist(z, z).pow(2)

    # Median pairwise squared distance (excluding diagonal)
    with torch.no_grad():
        mask = torch.triu(
            torch.ones_like(d2, dtype=torch.bool),
            diagonal=1
        )
        median_d2 = d2[mask].median().clamp_min(1e-8)

    bandwidths = [
        0.5 * median_d2,
        1.0 * median_d2,
        2.0 * median_d2
    ]

    K = sum(torch.exp(-d2 / bw) for bw in bandwidths)

    ns = source.size(0)

    K_ss = K[:ns, :ns]
    K_tt = K[ns:, ns:]
    K_st = K[:ns, ns:]

    return K_ss.mean() + K_tt.mean() - 2 * K_st.mean()


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------
def evaluate_dan(loader):
    dan_model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = dan_model(x)

            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())

    acc = np.mean(np.array(preds) == np.array(labels))
    f1 = f1_score(labels, preds, average="macro")

    return acc, f1


# ------------------------------------------------------------
# Train DAN
# ------------------------------------------------------------
best_f1 = -1
best_state = None
bad_epochs = 0

for epoch in range(30):

    dan_model.train()

    source_iters = [cycle(loader) for loader in source_loaders]
    target_iter = cycle(target_train_loader)

    steps = max(len(loader) for loader in source_loaders)

    for _ in range(steps):

        xs, ys = [], []

        # 8 examples from each source domain
        for src_iter in source_iters:
            x, y = next(src_iter)
            xs.append(x)
            ys.append(y)

        xs = torch.cat(xs).to(device)   # 24 source
        ys = torch.cat(ys).to(device)

        xt, _ = next(target_iter)
        xt = xt.to(device)              # 24 target

        optimizer.zero_grad()

        # Source logits + 512-D source features
        source_logits, source_features = dan_model(
            xs, return_features=True
        )

        # 512-D target features
        _, target_features = dan_model(
            xt, return_features=True
        )

        cls_loss = criterion(source_logits, ys)
        mmd = mmd_loss(source_features, target_features)

        # lambda_MMD = 1
        loss = cls_loss + mmd

        loss.backward()
        optimizer.step()

    # Source-validation macro-F1
    val_acc, val_f1 = evaluate_dan(val_loader_source)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f} | "
        f"MMD: {mmd.item():.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(dan_model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= 5:
        print("Early stopping.")
        break


# ------------------------------------------------------------
# Restore and save best DAN checkpoint
# ------------------------------------------------------------
dan_model.load_state_dict(best_state)

torch.save(
    dan_model.state_dict(),
    "dan-mmd.pt"
)

print("Saved checkpoint: dan-mmd.pt")

Epoch 01 | Val Acc: 0.8895 | Val Macro-F1: 0.8840 | MMD: 0.0898
Epoch 02 | Val Acc: 0.9176 | Val Macro-F1: 0.9172 | MMD: 0.0998
Epoch 03 | Val Acc: 0.9200 | Val Macro-F1: 0.9168 | MMD: 0.0842
Epoch 04 | Val Acc: 0.9143 | Val Macro-F1: 0.9133 | MMD: 0.0745
Epoch 05 | Val Acc: 0.9291 | Val Macro-F1: 0.9263 | MMD: 0.0718
Epoch 06 | Val Acc: 0.9233 | Val Macro-F1: 0.9221 | MMD: 0.0548
Epoch 07 | Val Acc: 0.9200 | Val Macro-F1: 0.9196 | MMD: 0.0759
Epoch 08 | Val Acc: 0.9060 | Val Macro-F1: 0.9020 | MMD: 0.0776
Epoch 09 | Val Acc: 0.8961 | Val Macro-F1: 0.8941 | MMD: 0.0663
Epoch 10 | Val Acc: 0.9143 | Val Macro-F1: 0.9117 | MMD: 0.0862
Early stopping.
Saved checkpoint: dan-mmd.pt


In [9]:
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
import torch


# ------------------------------------------------------------
# Load best DAN checkpoint
# ------------------------------------------------------------
dan_model.load_state_dict(
    torch.load(
        "dan-mmd.pt",
        map_location=device
    )
)

dan_model.eval()


# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------
def evaluate(loader):
    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in loader:

            x = x.to(
                device,
                non_blocking=True
            )

            # Only class predictions are needed.
            # No MMD is computed during evaluation.
            logits = dan_model(x)

            preds = logits.argmax(
                dim=1
            ).cpu()

            y_true.extend(
                y.cpu().numpy()
            )

            y_pred.extend(
                preds.numpy()
            )

    acc = accuracy_score(
        y_true,
        y_pred
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return acc, f1


# ------------------------------------------------------------
# Source validation domains
# ------------------------------------------------------------
print("Source Validation Results")
print("-" * 55)

source_results = []

for domain, val_set in zip(
    SOURCE_DOMAINS,
    source_val_sets
):

    loader = DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    acc, f1 = evaluate(loader)

    source_results.append(
        (domain, acc, f1)
    )

    print(
        f"{domain:15s} | "
        f"Accuracy: {acc:.4f} | "
        f"Macro-F1: {f1:.4f}"
    )


# ------------------------------------------------------------
# Mean source-validation performance
# ------------------------------------------------------------
mean_source_acc = sum(
    result[1] for result in source_results
) / len(source_results)

mean_source_f1 = sum(
    result[2] for result in source_results
) / len(source_results)

print("-" * 55)

print(
    f"{'Mean':15s} | "
    f"Accuracy: {mean_source_acc:.4f} | "
    f"Macro-F1: {mean_source_f1:.4f}"
)


# ------------------------------------------------------------
# Target domain
# ------------------------------------------------------------
target_eval_loader = DataLoader(
    target_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

target_acc, target_f1 = evaluate(
    target_eval_loader
)


# ------------------------------------------------------------
# Target results
# ------------------------------------------------------------
print("\nTarget Results")
print("-" * 55)

print(
    f"Accuracy : {target_acc:.4f}"
)

print(
    f"Macro-F1 : {target_f1:.4f}"
)

Source Validation Results
-------------------------------------------------------


/tmp/ipykernel_63539/2507806464.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


photo           | Accuracy: 0.9521 | Macro-F1: 0.9427
art_painting    | Accuracy: 0.9024 | Macro-F1: 0.9007
cartoon         | Accuracy: 0.9360 | Macro-F1: 0.9321
-------------------------------------------------------
Mean            | Accuracy: 0.9302 | Macro-F1: 0.9252

Target Results
-------------------------------------------------------
Accuracy : 0.6895
Macro-F1 : 0.6588


# DANN - Adversarial Alignment

In [10]:
import copy
import math
from itertools import cycle
from sklearn.metrics import f1_score
from torch.autograd import Function
from torchvision import models

# ------------------------------------------------------------
# Gradient Reversal Layer
# ------------------------------------------------------------
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


def grad_reverse(x, alpha):
    return GradientReversal.apply(x, alpha)


# ------------------------------------------------------------
# DANN Model
# ------------------------------------------------------------
class DANNPACSResNet18(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        weights = ResNet18_Weights.IMAGENET1K_V1
        backbone = resnet18(weights=weights)


        # 512-D feature extractor
        self.features = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        # Class classifier
        self.classifier = nn.Linear(512, num_classes)

        # Domain discriminator: 512 -> 512 -> 2
        self.domain_classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward_features(self, x):
        x = self.features(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        features = self.forward_features(x)
        return self.classifier(features)

    def forward_domain(self, features, alpha):
        features = grad_reverse(features, alpha)
        return self.domain_classifier(features)


# ------------------------------------------------------------
# Fresh model
# ------------------------------------------------------------
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dann_model = DANNPACSResNet18(num_classes=7).to(device)


# 8 examples from each source domain
source_loaders = [
    DataLoader(
        ds,
        batch_size=8,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    for ds in source_train_sets
]

# 24 target examples
target_train_loader = DataLoader(
    target_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    dann_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------
def evaluate_dann(loader):
    dann_model.eval()

    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = dann_model(x)

            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())

    acc = np.mean(np.array(preds) == np.array(labels))
    f1 = f1_score(labels, preds, average="macro")

    return acc, f1


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
best_f1 = -1
best_state = None
bad_epochs = 0

max_epochs = 30
steps_per_epoch = max(len(loader) for loader in source_loaders)
total_steps = max_epochs * steps_per_epoch

global_step = 0

for epoch in range(max_epochs):

    dann_model.train()

    source_iters = [cycle(loader) for loader in source_loaders]
    target_iter = cycle(target_train_loader)

    for _ in range(steps_per_epoch):

        # ------------------------------------
        # 24 source examples
        # ------------------------------------
        xs, ys = [], []

        for src_iter in source_iters:
            x, y = next(src_iter)
            xs.append(x)
            ys.append(y)

        xs = torch.cat(xs).to(device)
        ys = torch.cat(ys).to(device)

        # ------------------------------------
        # 24 target examples
        # ------------------------------------
        xt, _ = next(target_iter)
        xt = xt.to(device)

        # ------------------------------------
        # GRL schedule
        # ------------------------------------
        p = global_step / max(total_steps - 1, 1)
        alpha = 2.0 / (1.0 + math.exp(-10 * p)) - 1.0

        optimizer.zero_grad()

        # ------------------------------------
        # Features
        # ------------------------------------
        fs = dann_model.forward_features(xs)
        ft = dann_model.forward_features(xt)

        # Source classification
        class_logits = dann_model.classifier(fs)
        class_loss = class_criterion(class_logits, ys)

        # ------------------------------------
        # Domain classification
        # source = 0, target = 1
        # ------------------------------------
        features = torch.cat([fs, ft], dim=0)

        domain_labels = torch.cat([
            torch.zeros(fs.size(0), dtype=torch.long),
            torch.ones(ft.size(0), dtype=torch.long)
        ]).to(device)

        domain_logits = dann_model.forward_domain(
            features, alpha
        )

        domain_loss = domain_criterion(
            domain_logits,
            domain_labels
        )

        # Unit weight for domain loss
        loss = class_loss + domain_loss

        loss.backward()
        optimizer.step()

        global_step += 1


    # --------------------------------------------------------
    # Source validation
    # --------------------------------------------------------
    val_acc, val_f1 = evaluate_dann(val_loader_source)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Cls: {class_loss.item():.4f} | "
        f"Domain: {domain_loss.item():.4f} | "
        f"alpha: {alpha:.3f}"
    )

    # Early stopping
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(dann_model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= 5:
        print("Early stopping.")
        break


# ------------------------------------------------------------
# Restore + save best DANN
# ------------------------------------------------------------
dann_model.load_state_dict(best_state)

torch.save(
    dann_model.state_dict(),
    "dann-adversarial.pt"
)

print("Saved checkpoint: dann-adversarial.pt")

Epoch 01 | Val Acc: 0.8574 | Val F1: 0.8544 | Cls: 0.2943 | Domain: 0.5824 | alpha: 0.164
Epoch 02 | Val Acc: 0.8697 | Val F1: 0.8672 | Cls: 0.0345 | Domain: 0.6831 | alpha: 0.321
Epoch 03 | Val Acc: 0.8417 | Val F1: 0.8359 | Cls: 0.0246 | Domain: 0.7421 | alpha: 0.462
Epoch 04 | Val Acc: 0.8739 | Val F1: 0.8643 | Cls: 0.0606 | Domain: 0.7114 | alpha: 0.582
Epoch 05 | Val Acc: 0.8969 | Val F1: 0.8978 | Cls: 0.0111 | Domain: 0.7153 | alpha: 0.682
Epoch 06 | Val Acc: 0.8673 | Val F1: 0.8583 | Cls: 0.0020 | Domain: 0.7069 | alpha: 0.761
Epoch 07 | Val Acc: 0.8524 | Val F1: 0.8423 | Cls: 0.0051 | Domain: 0.6926 | alpha: 0.823
Epoch 08 | Val Acc: 0.8244 | Val F1: 0.8180 | Cls: 0.0036 | Domain: 0.7179 | alpha: 0.870
Epoch 09 | Val Acc: 0.7667 | Val F1: 0.7799 | Cls: 0.0818 | Domain: 0.6990 | alpha: 0.905
Epoch 10 | Val Acc: 0.8961 | Val F1: 0.8976 | Cls: 0.0037 | Domain: 0.6878 | alpha: 0.931
Early stopping.
Saved checkpoint: dann-adversarial.pt


In [11]:
from sklearn.metrics import accuracy_score, f1_score

# Load best DANN checkpoint
dann_model.load_state_dict(
    torch.load("dann-adversarial.pt", map_location=device)
)
dann_model.eval()


def evaluate(loader):
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            # Only class predictions are needed during evaluation
            logits = dann_model(x)
            preds = logits.argmax(dim=1).cpu()

            y_true.extend(y.numpy())
            y_pred.extend(preds.numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")

    return acc, f1


# Source validation domains
print("Source Validation Results")
print("-" * 45)

for domain, val_set in zip(SOURCE_DOMAINS, source_val_sets):
    loader = DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    acc, f1 = evaluate(loader)

    print(
        f"{domain:12s} | "
        f"Accuracy: {acc:.4f} | "
        f"Macro-F1: {f1:.4f}"
    )


# Target domain
target_eval_loader = DataLoader(
    target_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

target_acc, target_f1 = evaluate(target_eval_loader)

print("\nTarget Results")
print("-" * 45)
print(f"Accuracy : {target_acc:.4f}")
print(f"Macro-F1 : {target_f1:.4f}")

Source Validation Results
---------------------------------------------


/tmp/ipykernel_63539/2893576991.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("dann-adversarial.pt", map_location=device)


photo        | Accuracy: 0.9341 | Macro-F1: 0.9305
art_painting | Accuracy: 0.8463 | Macro-F1: 0.8473
cartoon      | Accuracy: 0.9147 | Macro-F1: 0.9178

Target Results
---------------------------------------------
Accuracy : 0.7015
Macro-F1 : 0.6431


# CDAN – Class-Conditional Adversarial Alignment

In [12]:
import copy
import math
from itertools import cycle
from sklearn.metrics import f1_score
from torch.autograd import Function


# ------------------------------------------------------------
# Gradient Reversal Layer
# ------------------------------------------------------------
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


def grad_reverse(x, alpha):
    return GradientReversal.apply(x, alpha)


# ------------------------------------------------------------
# CDAN Model
# ------------------------------------------------------------
class CDANPACSResNet18(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        # 512-D feature extractor
        self.features = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        # Classifier
        self.classifier = nn.Linear(512, num_classes)

        # CDAN discriminator input = 512 x 7 = 3584
        self.domain_classifier = nn.Sequential(
            nn.Linear(512 * num_classes, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward_features(self, x):
        x = self.features(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        f = self.forward_features(x)
        return self.classifier(f)

    def forward_domain(self, f, p, alpha):
        # Outer product f ⊗ p
        # [B, 512, 1] x [B, 1, 7] -> [B, 512, 7]
        g = torch.bmm(
            f.unsqueeze(2),
            p.unsqueeze(1)
        ).flatten(1)

        g = grad_reverse(g, alpha)
        return self.domain_classifier(g)


# ------------------------------------------------------------
# Fresh CDAN model
# ------------------------------------------------------------
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cdan_model = CDANPACSResNet18(num_classes=7).to(device)


# 8 examples from each source domain = 24 source
source_loaders = [
    DataLoader(
        ds,
        batch_size=8,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    for ds in source_train_sets
]

# 24 target examples
target_train_loader = DataLoader(
    target_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    cdan_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
def evaluate_cdan(loader):
    cdan_model.eval()

    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = cdan_model(x)

            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())

    acc = np.mean(np.array(preds) == np.array(labels))
    f1 = f1_score(labels, preds, average="macro")

    return acc, f1


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
best_f1 = -1
best_state = None
bad_epochs = 0

max_epochs = 30
steps_per_epoch = max(len(loader) for loader in source_loaders)
total_steps = max_epochs * steps_per_epoch

global_step = 0

for epoch in range(max_epochs):

    cdan_model.train()

    source_iters = [cycle(loader) for loader in source_loaders]
    target_iter = cycle(target_train_loader)

    for _ in range(steps_per_epoch):

        # -----------------------------
        # 24 source examples
        # -----------------------------
        xs, ys = [], []

        for src_iter in source_iters:
            x, y = next(src_iter)
            xs.append(x)
            ys.append(y)

        xs = torch.cat(xs).to(device)
        ys = torch.cat(ys).to(device)

        # -----------------------------
        # 24 target examples
        # -----------------------------
        xt, _ = next(target_iter)
        xt = xt.to(device)

        # -----------------------------
        # GRL schedule
        # -----------------------------
        p_progress = global_step / max(total_steps - 1, 1)

        alpha = (
            2.0 / (1.0 + math.exp(-10 * p_progress))
            - 1.0
        )

        optimizer.zero_grad()

        # -----------------------------
        # Source
        # -----------------------------
        fs = cdan_model.forward_features(xs)
        source_logits = cdan_model.classifier(fs)
        ps = torch.softmax(source_logits, dim=1)

        # Classification loss: source only
        class_loss = class_criterion(
            source_logits,
            ys
        )

        # -----------------------------
        # Target
        # -----------------------------
        ft = cdan_model.forward_features(xt)
        target_logits = cdan_model.classifier(ft)
        pt = torch.softmax(target_logits, dim=1)

        # IMPORTANT:
        # Neither features nor probabilities are detached.

        # -----------------------------
        # CDAN domain predictions
        # -----------------------------
        source_domain_logits = cdan_model.forward_domain(
            fs, ps, alpha
        )

        target_domain_logits = cdan_model.forward_domain(
            ft, pt, alpha
        )

        domain_logits = torch.cat([
            source_domain_logits,
            target_domain_logits
        ])

        # source = 0, target = 1
        domain_labels = torch.cat([
            torch.zeros(
                fs.size(0),
                dtype=torch.long,
                device=device
            ),
            torch.ones(
                ft.size(0),
                dtype=torch.long,
                device=device
            )
        ])

        domain_loss = domain_criterion(
            domain_logits,
            domain_labels
        )

        # Unit domain-loss weight
        loss = class_loss + domain_loss

        loss.backward()
        optimizer.step()

        global_step += 1


    # --------------------------------------------------------
    # Source validation
    # --------------------------------------------------------
    val_acc, val_f1 = evaluate_cdan(
        val_loader_source
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Cls: {class_loss.item():.4f} | "
        f"Domain: {domain_loss.item():.4f} | "
        f"alpha: {alpha:.3f}"
    )

    # Early stopping
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(
            cdan_model.state_dict()
        )
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= 5:
        print("Early stopping.")
        break


# ------------------------------------------------------------
# Restore and save best CDAN
# ------------------------------------------------------------
cdan_model.load_state_dict(best_state)

torch.save(
    cdan_model.state_dict(),
    "cdan-adversarial.pt"
)

print("Saved checkpoint: cdan-adversarial.pt")

Epoch 01 | Val Acc: 0.8895 | Val F1: 0.8858 | Cls: 0.2089 | Domain: 0.5869 | alpha: 0.164
Epoch 02 | Val Acc: 0.8945 | Val F1: 0.8913 | Cls: 0.0973 | Domain: 0.7603 | alpha: 0.321
Epoch 03 | Val Acc: 0.8747 | Val F1: 0.8715 | Cls: 0.0985 | Domain: 0.7007 | alpha: 0.462
Epoch 04 | Val Acc: 0.8706 | Val F1: 0.8730 | Cls: 0.0384 | Domain: 0.7285 | alpha: 0.582
Epoch 05 | Val Acc: 0.8541 | Val F1: 0.8531 | Cls: 0.0172 | Domain: 0.6684 | alpha: 0.682
Epoch 06 | Val Acc: 0.8945 | Val F1: 0.8989 | Cls: 0.0781 | Domain: 0.7438 | alpha: 0.761
Epoch 07 | Val Acc: 0.9002 | Val F1: 0.9008 | Cls: 0.0036 | Domain: 0.7317 | alpha: 0.823
Epoch 08 | Val Acc: 0.8739 | Val F1: 0.8713 | Cls: 0.0560 | Domain: 0.7326 | alpha: 0.870
Epoch 09 | Val Acc: 0.8747 | Val F1: 0.8716 | Cls: 0.0906 | Domain: 0.6834 | alpha: 0.905
Epoch 10 | Val Acc: 0.8879 | Val F1: 0.8874 | Cls: 0.0191 | Domain: 0.6888 | alpha: 0.931
Epoch 11 | Val Acc: 0.8871 | Val F1: 0.8893 | Cls: 0.0016 | Domain: 0.6794 | alpha: 0.950
Epoch 12 |

# Results of Each Backbone

In [13]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = model(x)
            preds = logits.argmax(dim=1).cpu()

            y_true.extend(y.numpy())
            y_pred.extend(preds.numpy())

    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro")
    )


# ------------------------------------------------------------
# Models + checkpoints
# ------------------------------------------------------------
models_to_evaluate = {
    "Source-only": (
        PACSResNet18(num_classes=7).to(device),
        "erm-baseline.pt"
    ),

    "DAN": (
        PACSResNet18(num_classes=7).to(device),
        "dan-mmd.pt"
    ),

    "DANN": (
        DANNPACSResNet18(num_classes=7).to(device),
        "dann-adversarial.pt"
    ),

    "CDAN": (
        CDANPACSResNet18(num_classes=7).to(device),
        "cdan-adversarial.pt"
    )
}


results = []

for name, (model, checkpoint) in models_to_evaluate.items():

    # Load fixed checkpoint
    model.load_state_dict(
        torch.load(checkpoint, map_location=device)
    )

    # Source validation
    src_acc, src_f1 = evaluate_model(
        model, val_loader_source
    )

    # Target
    tgt_acc, tgt_f1 = evaluate_model(
        model, target_loader
    )

    results.append({
        "Method": name,
        "Source Val Acc": src_acc,
        "Source Val F1": src_f1,
        "Target Acc": tgt_acc,
        "Target F1": tgt_f1
    })


# ------------------------------------------------------------
# Target accuracy change relative to Source-only
# ------------------------------------------------------------

erm_target_acc = results[0]["Target Acc"]

for r in results:
    r["Target Acc Δ vs Source-only (pp)"] = (
        (r["Target Acc"] - erm_target_acc) * 100
    )

# ------------------------------------------------------------
# Convert results list to DataFrame
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Display results without pandas Styler / jinja2
# ------------------------------------------------------------

display_df = results_df.copy()

for col in [
    "Source Val Acc",
    "Source Val F1",
    "Target Acc",
    "Target F1"
]:
    display_df[col] = display_df[col].map(
        lambda x: f"{x * 100:.2f}%"
    )

display_df["Target Acc Δ vs Source-only (pp)"] = (
    display_df["Target Acc Δ vs Source-only (pp)"]
    .map(lambda x: f"{x:+.2f}")
)

print("Final Comparison")
print("-" * 90)

display(display_df)

/tmp/ipykernel_63539/3397680194.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(checkpoint, map_location=device)
/tmp/ipykernel_63539/3397680194.py:57: Futur

Final Comparison
------------------------------------------------------------------------------------------


,Method,Source Val Acc,Source Val F1,Target Acc,Target F1,Target Acc Δ vs Source-only (pp)
0,Source-only,94.06%,94.16%,61.03%,59.48%,+0.00
1,DAN,92.91%,92.63%,68.95%,65.88%,+7.92
2,DANN,89.69%,89.78%,70.15%,64.31%,+9.11
3,CDAN,90.02%,90.08%,65.05%,59.85%,+4.02


# Evaluation and Alignment Diagnostic

In [14]:
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split


# ============================================================
# 1. Create Source-vs-Target dataset
#
# Each sample:
#     Xi = image
#     Yi = domain label
#
#     source -> 0
#     target -> 1
#
# Existing PACS class labels (0-6) are ignored.
# ============================================================

class DomainDataset(Dataset):
    def __init__(self, source_dataset, target_dataset):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset

    def __len__(self):
        return len(self.source_dataset) + len(self.target_dataset)

    def __getitem__(self, idx):

        # -------------------------
        # Source image
        # -------------------------
        if idx < len(self.source_dataset):
            image, _ = self.source_dataset[idx]

            # Domain label
            domain_label = 0

        # -------------------------
        # Target image
        # -------------------------
        else:
            target_idx = idx - len(self.source_dataset)

            image, _ = self.target_dataset[target_idx]

            # Domain label
            domain_label = 1

        return image, domain_label


# Use datasets ALREADY defined in your notebook.
#
# source_test_dataset:
#     photo + art_painting + cartoon
#
# target_dataset:
#     sketch

domain_dataset = DomainDataset(
    source_test_dataset,
    target_dataset
)


# ============================================================
# 2. Construct labels for stratification
# ============================================================

domain_labels = np.concatenate([
    np.zeros(
        len(source_test_dataset),
        dtype=np.int64
    ),
    np.ones(
        len(target_dataset),
        dtype=np.int64
    )
])

indices = np.arange(len(domain_dataset))


# ============================================================
# 3. 70 / 30 STRATIFIED split
# ============================================================
SEED = 6304
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=domain_labels
)


train_domain_dataset = Subset(
    domain_dataset,
    train_idx
)

test_domain_dataset = Subset(
    domain_dataset,
    test_idx
)


# ============================================================
# 4. DataLoaders
# ============================================================

train_domain_loader = DataLoader(
    train_domain_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_domain_loader = DataLoader(
    test_domain_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# Sanity checks
# ============================================================

print("Complete domain dataset")
print("-" * 50)
print(f"Source (Y=0): {np.sum(domain_labels == 0)}")
print(f"Target (Y=1): {np.sum(domain_labels == 1)}")
print(f"Total       : {len(domain_dataset)}")

print("\n70/30 split")
print("-" * 50)

print(f"Train: {len(train_idx)}")
print(
    f"  Source: {np.sum(domain_labels[train_idx] == 0)}"
)
print(
    f"  Target: {np.sum(domain_labels[train_idx] == 1)}"
)

print(f"\nTest: {len(test_idx)}")
print(
    f"  Source: {np.sum(domain_labels[test_idx] == 0)}"
)
print(
    f"  Target: {np.sum(domain_labels[test_idx] == 1)}"
)

Complete domain dataset
--------------------------------------------------
Source (Y=0): 6062
Target (Y=1): 3929
Total       : 9991

70/30 split
--------------------------------------------------
Train: 6993
  Source: 4243
  Target: 2750

Test: 2998
  Source: 1819
  Target: 1179


In [15]:
# ============================================================
# Extract 512-D feature maps from all four trained models
# ERM, DAN, DANN, CDAN
# ============================================================

import torch
import numpy as np

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 1. CREATE FRESH INSTANCES OF THE CORRECT MODEL CLASSES
# ============================================================

# ERM and DAN use PACSResNet18
erm_feature_model = PACSResNet18(
    num_classes=7
).to(device)

dan_feature_model = PACSResNet18(
    num_classes=7
).to(device)

# DANN has its own architecture
dann_feature_model = DANNPACSResNet18(
    num_classes=7
).to(device)

# CDAN has its own architecture
cdan_feature_model = CDANPACSResNet18(
    num_classes=7
).to(device)


# ============================================================
# 2. LOAD THE CORRECT CHECKPOINT INTO EACH MODEL
# ============================================================

erm_feature_model.load_state_dict(
    torch.load(
        "erm-baseline.pt",
        map_location=device
    )
)

dan_feature_model.load_state_dict(
    torch.load(
        "dan-mmd.pt",
        map_location=device
    )
)

dann_feature_model.load_state_dict(
    torch.load(
        "dann-adversarial.pt",
        map_location=device
    )
)

cdan_feature_model.load_state_dict(
    torch.load(
        "cdan-adversarial.pt",
        map_location=device
    )
)


# Evaluation mode
erm_feature_model.eval()
dan_feature_model.eval()
dann_feature_model.eval()
cdan_feature_model.eval()

print("\nAll four checkpoints loaded successfully.")


# ============================================================
# 3. MODEL-SPECIFIC FEATURE EXTRACTION
# ============================================================

def get_512_features(net, images, model_type):
    """
    Returns the final 512-D representation immediately
    before the class classifier.
    """

    # --------------------------------------------------------
    # ERM
    # PACSResNet18:
    #
    # logits, features =
    #     model(x, return_features=True)
    # --------------------------------------------------------
    if model_type == "ERM":

        _, features = net(
            images,
            return_features=True
        )

    # --------------------------------------------------------
    # DAN
    # DAN also uses PACSResNet18
    # --------------------------------------------------------
    elif model_type == "DAN":

        _, features = net(
            images,
            return_features=True
        )

    # --------------------------------------------------------
    # DANN
    # DANNPACSResNet18:
    #
    # features = model.forward_features(x)
    # --------------------------------------------------------
    elif model_type == "DANN":

        features = net.forward_features(
            images
        )

    # --------------------------------------------------------
    # CDAN
    # CDANPACSResNet18:
    #
    # features = model.forward_features(x)
    # --------------------------------------------------------
    elif model_type == "CDAN":

        features = net.forward_features(
            images
        )

    else:

        raise ValueError(
            f"Unknown model type: {model_type}"
        )

    # --------------------------------------------------------
    # Sanity check
    # --------------------------------------------------------

    if features.ndim != 2:
        raise RuntimeError(
            f"{model_type}: expected a 2-D feature matrix, "
            f"got shape {tuple(features.shape)}"
        )

    if features.shape[1] != 512:
        raise RuntimeError(
            f"{model_type}: expected 512 features, "
            f"got {features.shape[1]}"
        )

    return features


# ============================================================
# 4. EXTRACT AN ENTIRE DATASET
# ============================================================

def extract_features(
    net,
    loader,
    model_type
):

    feature_batches = []
    label_batches = []

    net.eval()

    with torch.no_grad():

        for images, domain_labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            features = get_512_features(
                net,
                images,
                model_type
            )

            feature_batches.append(
                features.detach().cpu()
            )

            label_batches.append(
                domain_labels.detach().cpu()
            )

    features = torch.cat(
        feature_batches,
        dim=0
    )

    labels = torch.cat(
        label_batches,
        dim=0
    )

    return features, labels


# ============================================================
# 5. ALL FOUR MODELS
# ============================================================

models = {
    "ERM": erm_feature_model,
    "DAN": dan_feature_model,
    "DANN": dann_feature_model,
    "CDAN": cdan_feature_model
}


# ============================================================
# 6. EXTRACT TRAIN + TEST FEATURES
# ============================================================

feature_maps = {}

for model_name, net in models.items():

    print("\n" + "=" * 65)
    print(f"Extracting {model_name} features...")
    print("=" * 65)

    # TRAIN
    train_features, train_labels = extract_features(
        net,
        train_domain_loader,
        model_name
    )

    # TEST
    test_features, test_labels = extract_features(
        net,
        test_domain_loader,
        model_name
    )

    feature_maps[model_name] = {
        "train_features": train_features,
        "train_labels": train_labels,
        "test_features": test_features,
        "test_labels": test_labels
    }

    print(
        f"Train: {tuple(train_features.shape)}"
    )

    print(
        f"Test : {tuple(test_features.shape)}"
    )


# ============================================================
# 7. PRINT FEATURE MAP DETAILS
# ============================================================

def print_feature_details(
    model_name,
    split_name,
    features,
    labels
):

    n_source = (
        labels == 0
    ).sum().item()

    n_target = (
        labels == 1
    ).sum().item()

    print("\n" + "-" * 65)
    print(f"{model_name} | {split_name}")
    print("-" * 65)

    print(
        f"Feature map shape   : {tuple(features.shape)}"
    )

    print(
        f"Number of images    : {features.shape[0]}"
    )

    print(
        f"Feature dimension   : {features.shape[1]}"
    )

    print(
        f"Source (Y=0)        : {n_source}"
    )

    print(
        f"Target (Y=1)        : {n_target}"
    )

    print(
        f"Feature dtype       : {features.dtype}"
    )

    print(
        f"Feature min         : {features.min().item():.6f}"
    )

    print(
        f"Feature max         : {features.max().item():.6f}"
    )

    print(
        f"Feature mean        : {features.mean().item():.6f}"
    )

    print(
        f"Feature std         : {features.std().item():.6f}"
    )


# ============================================================
# 8. DISPLAY DETAILS FOR ALL FOUR MODELS
# ============================================================

print("\n\n")
print("=" * 65)
print("FEATURE MAP SUMMARY")
print("=" * 65)

for model_name in models:

    data = feature_maps[model_name]

    print_feature_details(
        model_name,
        "TRAIN",
        data["train_features"],
        data["train_labels"]
    )

    print_feature_details(
        model_name,
        "TEST",
        data["test_features"],
        data["test_labels"]
    )


# ============================================================
# 9. SANITY CHECKS
# ============================================================

for model_name in models:

    data = feature_maps[model_name]

    assert data["train_features"].shape == (
        6993,
        512
    ), (
        f"{model_name}: unexpected train shape "
        f"{data['train_features'].shape}"
    )

    assert data["test_features"].shape == (
        2998,
        512
    ), (
        f"{model_name}: unexpected test shape "
        f"{data['test_features'].shape}"
    )

    assert (
        data["train_labels"] == 0
    ).sum().item() == 4243

    assert (
        data["train_labels"] == 1
    ).sum().item() == 2750

    assert (
        data["test_labels"] == 0
    ).sum().item() == 1819

    assert (
        data["test_labels"] == 1
    ).sum().item() == 1179


print("\n" + "=" * 65)
print("All feature extraction sanity checks passed.")
print("=" * 65)

Device: cuda

All four checkpoints loaded successfully.

Extracting ERM features...


/tmp/ipykernel_63539/3739404034.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(
/tmp/ipykernel_63539/3739404034.py:52: FutureWarning: You are using `torch.l

Train: (6993, 512)
Test : (2998, 512)

Extracting DAN features...
Train: (6993, 512)
Test : (2998, 512)

Extracting DANN features...
Train: (6993, 512)
Test : (2998, 512)

Extracting CDAN features...
Train: (6993, 512)
Test : (2998, 512)



FEATURE MAP SUMMARY

-----------------------------------------------------------------
ERM | TRAIN
-----------------------------------------------------------------
Feature map shape   : (6993, 512)
Number of images    : 6993
Feature dimension   : 512
Source (Y=0)        : 4243
Target (Y=1)        : 2750
Feature dtype       : torch.float32
Feature min         : 0.000000
Feature max         : 11.798775
Feature mean        : 0.744845
Feature std         : 0.836598

-----------------------------------------------------------------
ERM | TEST
-----------------------------------------------------------------
Feature map shape   : (2998, 512)
Number of images    : 2998
Feature dimension   : 512
Source (Y=0)        : 1819
Target (Y=1)        : 1179
Feature

In [16]:
# ============================================================
# Logistic Regression Domain Classification
#
# Xi = 512-D image representation
# Yi = domain label
#      0 -> Source
#      1 -> Target
#
# Four independent logistic regression models:
#   1. ERM features
#   2. DAN features
#   3. DANN features
#   4. CDAN features
# ============================================================

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# Storage
# ============================================================

logistic_models = {}
logistic_results = {}


# ============================================================
# Train one Logistic Regression model for each representation
# ============================================================

for model_name in ["ERM", "DAN", "DANN", "CDAN"]:

    print("\n" + "=" * 70)
    print(f"{model_name} FEATURES -> LOGISTIC REGRESSION")
    print("=" * 70)

    # --------------------------------------------------------
    # Xi: 512-D image representations
    # --------------------------------------------------------

    X_train = (
        feature_maps[model_name]["train_features"]
        .numpy()
    )

    X_test = (
        feature_maps[model_name]["test_features"]
        .numpy()
    )

    # --------------------------------------------------------
    # Yi: Domain labels
    #
    # 0 = source
    # 1 = target
    # --------------------------------------------------------

    y_train = (
        feature_maps[model_name]["train_labels"]
        .numpy()
    )

    y_test = (
        feature_maps[model_name]["test_labels"]
        .numpy()
    )


    # --------------------------------------------------------
    # Print input information
    # --------------------------------------------------------

    print("\nDataset information")
    print("-" * 45)

    print(
        f"X_train shape : {X_train.shape}"
    )

    print(
        f"y_train shape : {y_train.shape}"
    )

    print(
        f"X_test shape  : {X_test.shape}"
    )

    print(
        f"y_test shape  : {y_test.shape}"
    )

    print(
        f"Train source  : {(y_train == 0).sum()}"
    )

    print(
        f"Train target  : {(y_train == 1).sum()}"
    )

    print(
        f"Test source   : {(y_test == 0).sum()}"
    )

    print(
        f"Test target   : {(y_test == 1).sum()}"
    )


    # ========================================================
    # Logistic Regression
    # ========================================================

    domain_classifier = LogisticRegression(
        max_iter=2000,
        random_state=6304
    )


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    domain_classifier.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    y_pred = domain_classifier.predict(
        X_test
    )

    y_prob = domain_classifier.predict_proba(
        X_test
    )[:, 1]


    # ========================================================
    # Evaluation
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )


    # ========================================================
    # Save trained Logistic Regression model
    # ========================================================

    logistic_models[model_name] = domain_classifier


    # ========================================================
    # Save results
    # ========================================================

    logistic_results[model_name] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
        "y_true": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob
    }


    # ========================================================
    # Print results
    # ========================================================

    print("\nTest Results")
    print("-" * 45)

    print(
        f"Accuracy  : {accuracy:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1 Score  : {f1:.4f}"
    )


    print("\nConfusion Matrix")
    print("-" * 45)

    print(cm)


    print("\nClassification Report")
    print("-" * 45)

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "Source",
                "Target"
            ],
            digits=4,
            zero_division=0
        )
    )


# ============================================================
# FINAL COMPARISON
# ============================================================

print("\n\n")
print("=" * 78)
print("LOGISTIC REGRESSION DOMAIN CLASSIFICATION SUMMARY")
print("=" * 78)

print(
    f"{'Representation':<18}"
    f"{'Accuracy':>12}"
    f"{'Precision':>12}"
    f"{'Recall':>12}"
    f"{'F1':>12}"
)

print("-" * 78)

for model_name in ["ERM", "DAN", "DANN", "CDAN"]:

    result = logistic_results[model_name]

    print(
        f"{model_name:<18}"
        f"{result['accuracy']:>12.4f}"
        f"{result['precision']:>12.4f}"
        f"{result['recall']:>12.4f}"
        f"{result['f1']:>12.4f}"
    )


ERM FEATURES -> LOGISTIC REGRESSION

Dataset information
---------------------------------------------
X_train shape : (6993, 512)
y_train shape : (6993,)
X_test shape  : (2998, 512)
y_test shape  : (2998,)
Train source  : 4243
Train target  : 2750
Test source   : 1819
Test target   : 1179

Test Results
---------------------------------------------
Accuracy  : 0.9987
Precision : 0.9966
Recall    : 1.0000
F1 Score  : 0.9983

Confusion Matrix
---------------------------------------------
[[1815    4]
 [   0 1179]]

Classification Report
---------------------------------------------
              precision    recall  f1-score   support

      Source     1.0000    0.9978    0.9989      1819
      Target     0.9966    1.0000    0.9983      1179

    accuracy                         0.9987      2998
   macro avg     0.9983    0.9989    0.9986      2998
weighted avg     0.9987    0.9987    0.9987      2998


DAN FEATURES -> LOGISTIC REGRESSION

Dataset information
---------------------------

# Controlled Design Study

In [17]:
# ============================================================
# 6. Controlled Design Study: DAN lambda_MMD sweep
#    lambda_MMD in {0.1, 1, 10}; every other setting fixed.
# ============================================================

import copy
from itertools import cycle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

LAMBDA_MMD_VALUES = [0.1, 1.0, 10.0]



def evaluate_classifier(net, loader):
    net.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = net(x)
            y_true.extend(y.numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())
    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro")
    )


def extract_domain_features(net, loader):
    net.eval()
    features, labels = [], []
    with torch.no_grad():
        for x, domain_y in loader:
            x = x.to(device)
            _, feat = net(x, return_features=True)
            features.append(feat.cpu())
            labels.append(domain_y.cpu())
    return torch.cat(features).numpy(), torch.cat(labels).numpy()


def train_dan_for_lambda(lambda_mmd):
    # Reset RNG so initialization and data-order randomness are matched across runs.
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    net = PACSResNet18(num_classes=7).to(device)

    # Same batches as the original DAN experiment: 8 x 3 = 24 source, 24 target.
    src_loaders = []
    for i, ds in enumerate(source_train_sets):
        g = torch.Generator().manual_seed(SEED + i)
        src_loaders.append(DataLoader(
            ds, batch_size=8, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, generator=g
        ))

    target_g = torch.Generator().manual_seed(SEED + len(source_train_sets))
    tgt_train_loader = DataLoader(
        target_dataset, batch_size=24, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, generator=target_g
    )

    loss_fn = torch.nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-4)

    best_f1 = -1.0
    best_state = None
    bad_epochs = 0

    for epoch in range(30):
        net.train()
        src_iters = [cycle(loader) for loader in src_loaders]
        tgt_iter = cycle(tgt_train_loader)
        steps = max(len(loader) for loader in src_loaders)

        for _ in range(steps):
            xs, ys = [], []
            for src_iter in src_iters:
                x, y = next(src_iter)
                xs.append(x)
                ys.append(y)

            xs = torch.cat(xs).to(device)
            ys = torch.cat(ys).to(device)
            xt, _ = next(tgt_iter)
            xt = xt.to(device)

            opt.zero_grad()
            source_logits, source_features = net(xs, return_features=True)
            _, target_features = net(xt, return_features=True)

            cls_loss = loss_fn(source_logits, ys)
            align_loss = mmd_loss(source_features, target_features)
            loss = cls_loss + lambda_mmd * align_loss
            loss.backward()
            opt.step()

        # Early stopping is based ONLY on labeled source validation data.
        src_val_acc, src_val_f1 = evaluate_classifier(net, val_loader_source)
        print(
            f"lambda={lambda_mmd:g} | epoch={epoch+1:02d} | "
            f"source-val acc={src_val_acc:.4f} | source-val F1={src_val_f1:.4f}"
        )

        if src_val_f1 > best_f1:
            best_f1 = src_val_f1
            best_state = copy.deepcopy(net.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= 5:
            break

    net.load_state_dict(best_state)
    return net


controlled_results = []
controlled_dan_models = {}

for lambda_mmd in LAMBDA_MMD_VALUES:
    print("\n" + "=" * 72)
    print(f"Training controlled DAN run: lambda_MMD = {lambda_mmd:g}")
    print("=" * 72)

    net = train_dan_for_lambda(lambda_mmd)
    controlled_dan_models[lambda_mmd] = net
    torch.save(net.state_dict(), f"dan-mmd-lambda-{lambda_mmd:g}.pt")

    # Source performance and target recognition.
    src_acc, src_f1 = evaluate_classifier(net, val_loader_source)
    tgt_acc, tgt_f1 = evaluate_classifier(net, target_loader)

    # Domain separability: same 70/30 source-vs-target split created above.
    X_domain_train, y_domain_train = extract_domain_features(net, train_domain_loader)
    X_domain_test, y_domain_test = extract_domain_features(net, test_domain_loader)

    domain_clf = LogisticRegression(max_iter=2000, random_state=SEED)
    domain_clf.fit(X_domain_train, y_domain_train)
    domain_pred = domain_clf.predict(X_domain_test)
    domain_acc = accuracy_score(y_domain_test, domain_pred)
    domain_f1 = f1_score(y_domain_test, domain_pred)

    controlled_results.append({
        "lambda_MMD": lambda_mmd,
        "Source Val Acc": src_acc,
        "Source Val Macro-F1": src_f1,
        "Domain Classifier Acc": domain_acc,
        "Domain Classifier F1": domain_f1,
        "Target Acc (analysis only)": tgt_acc,
        "Target Macro-F1 (analysis only)": tgt_f1,
    })

controlled_dan_results = pd.DataFrame(controlled_results).sort_values("lambda_MMD")

print("\n" + "=" * 72)
print("CONTROLLED DAN STUDY RESULTS")
print("=" * 72)
display(controlled_dan_results.round(4))

# Interpretation is descriptive only; it does not select or revise a tested setting.
print("\nINTERPRETATION")
print("-" * 72)
for _, r in controlled_dan_results.iterrows():
    print(
        f"lambda={r['lambda_MMD']:g}: "
        f"source F1={r['Source Val Macro-F1']:.4f}, "
        f"domain-clf acc={r['Domain Classifier Acc']:.4f}, "
        f"target F1={r['Target Macro-F1 (analysis only)']:.4f}"
    )

first = controlled_dan_results.iloc[0]
last = controlled_dan_results.iloc[-1]
print("\nAcross the bounded sweep (0.1 -> 10):")
print(
    f"- Source macro-F1 change: "
    f"{last['Source Val Macro-F1'] - first['Source Val Macro-F1']:+.4f}."
)
print(
    f"- Domain-classifier accuracy change: "
    f"{last['Domain Classifier Acc'] - first['Domain Classifier Acc']:+.4f}. "
    "Lower values indicate less source/target separability."
)
print(
    f"- Target macro-F1 change (analysis only): "
    f"{last['Target Macro-F1 (analysis only)'] - first['Target Macro-F1 (analysis only)']:+.4f}."
)
print(
    "Read the three rows together: stronger MMD pressure is useful only insofar as "
    "reduced domain separability does not destroy class-discriminative structure. "
    "The target metrics are reported only to analyze that trade-off after the fixed "
    "three-run experiment; they were not used to tune or rerun anything."
)


Training controlled DAN run: lambda_MMD = 0.1
lambda=0.1 | epoch=01 | source-val acc=0.8780 | source-val F1=0.8777
lambda=0.1 | epoch=02 | source-val acc=0.9151 | source-val F1=0.9137
lambda=0.1 | epoch=03 | source-val acc=0.9167 | source-val F1=0.9158
lambda=0.1 | epoch=04 | source-val acc=0.8730 | source-val F1=0.8759
lambda=0.1 | epoch=05 | source-val acc=0.9068 | source-val F1=0.9056
lambda=0.1 | epoch=06 | source-val acc=0.8763 | source-val F1=0.8738
lambda=0.1 | epoch=07 | source-val acc=0.8838 | source-val F1=0.8882
lambda=0.1 | epoch=08 | source-val acc=0.9200 | source-val F1=0.9201
lambda=0.1 | epoch=09 | source-val acc=0.8895 | source-val F1=0.8907
lambda=0.1 | epoch=10 | source-val acc=0.8681 | source-val F1=0.8599
lambda=0.1 | epoch=11 | source-val acc=0.8887 | source-val F1=0.8904
lambda=0.1 | epoch=12 | source-val acc=0.8912 | source-val F1=0.8892
lambda=0.1 | epoch=13 | source-val acc=0.9110 | source-val F1=0.9058

Training controlled DAN run: lambda_MMD = 1
lambda=1 | 

,lambda_MMD,Source Val Acc,Source Val Macro-F1,Domain Classifier Acc,Domain Classifier F1,Target Acc (analysis only),Target Macro-F1 (analysis only)
0,0.1,0.9200,0.9201,0.999,0.9987,0.7081,0.6943
1,1.0,0.9225,0.9217,0.999,0.9987,0.6953,0.6909
2,10.0,0.2119,0.0500,0.944,0.9241,0.0407,0.0112



INTERPRETATION
------------------------------------------------------------------------
lambda=0.1: source F1=0.9201, domain-clf acc=0.9990, target F1=0.6943
lambda=1: source F1=0.9217, domain-clf acc=0.9990, target F1=0.6909
lambda=10: source F1=0.0500, domain-clf acc=0.9440, target F1=0.0112

Across the bounded sweep (0.1 -> 10):
- Source macro-F1 change: -0.8701.
- Domain-classifier accuracy change: -0.0550. Lower values indicate less source/target separability.
- Target macro-F1 change (analysis only): -0.6831.
Read the three rows together: stronger MMD pressure is useful only insofar as reduced domain separability does not destroy class-discriminative structure. The target metrics are reported only to analyze that trade-off after the fixed three-run experiment; they were not used to tune or rerun anything.


In [21]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

class_names = target_dataset.classes

domain_loaders = {
    domain: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    for domain, dataset in zip(SOURCE_DOMAINS, source_test_sets)
}
domain_loaders[TARGET_DOMAIN] = target_loader

method_models = {
    "ERM": erm_feature_model,
    "DAN": dan_feature_model,
    "DANN": dann_feature_model,
    "CDAN": cdan_feature_model
}

def get_logits(model, images, method):
    if method in ["ERM", "DAN"]:
        return model(images)

    features = model.forward_features(images)

    for attr in ["classifier", "class_classifier", "label_classifier"]:
        if hasattr(model, attr):
            return getattr(model, attr)(features)

    raise AttributeError(f"Classifier not found for {method}")


def per_class_accuracy(model, loader, method):
    model.eval()

    correct = np.zeros(len(class_names))
    total = np.zeros(len(class_names))

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            preds = get_logits(model, images, method).argmax(dim=1)

            for i in range(len(class_names)):
                mask = labels == i
                total[i] += mask.sum().item()
                correct[i] += (preds[mask] == labels[mask]).sum().item()

    return np.divide(
        correct,
        total,
        out=np.zeros_like(correct),
        where=total != 0
    ) * 100


results = []

for method, model in method_models.items():
    for domain, loader in domain_loaders.items():
        accuracies = per_class_accuracy(model, loader, method)

        row = {
            "Method": method,
            "Domain": domain
        }

        row.update({
            cls: acc
            for cls, acc in zip(class_names, accuracies)
        })

        results.append(row)

per_class_table = pd.DataFrame(results)
per_class_table = per_class_table.set_index(["Method", "Domain"])

display(per_class_table.round(2))

dog  elephant  giraffe  guitar  horse   house  person
Method Domain                                                               
ERM    photo         96.30     99.50    98.90  100.00  99.50  100.00  100.00
       art_painting  97.10     99.61    98.60  100.00  96.52   99.32   97.77
       cartoon       97.94     99.12    98.84   98.52  98.46  100.00   99.26
       sketch        11.01     94.46    84.33   84.05  43.38   76.25   33.12
DAN    photo         95.77     98.02    98.35  100.00  95.48  100.00  100.00
       art_painting  97.10     97.65    99.30   99.46  94.03  100.00   97.33
       cartoon       98.97     96.50    99.42   98.52  97.53   99.65   98.77
       sketch        34.46     91.35    68.79   81.25  68.50   87.50   78.75
DANN   photo         98.94     92.57    97.80   98.39  88.44  100.00   99.77
       art_painting  94.20     90.20    96.84   92.93  87.56   99.66   97.10
       cartoon       96.40     93.22   100.00   96.30  95.99  100.00   97.53
       sketch        50.91     80.95    64.41   89.97  79.90   68.75   15.62
CDAN   photo         91.53     98.51    98.90   98.39  96.98  100.00  100.00
       art_painting  89.45     97.65    99.30   95.65  93.53   99.32   90.42
       cartoon       93.57     98.91    99.71   95.56  94.75   99.65   96.30
       sketch        19.95     94.46    54.98   83.55  79.41  100.00   33.12